# xAI Grok 4.7 on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

xAI's Grok 4.7 reasons on every request, its reasoning is encrypted, and it reads
images. Unlike Grok 4.6 it is served on `bedrock-runtime` only, through a cross-Region
inference profile. Grok 4.6 has its own notebook, [`02-grok-4-6`](02-grok-4-6.ipynb),
and Grok 4.3 [`01-grok-4-3`](01-grok-4-3.ipynb).

| Model | `bedrock-mantle` ID | `bedrock-runtime` ID | Notes |
|---|---|---|---|
| Grok 4.7 | not served | `us.xai.grok-4.7` or `global.xai.grok-4.7` | `bedrock-runtime` only, through an inference profile |

IDs as of October 2026. `bedrock-runtime` reaches the model only through a cross-Region
inference profile: `us.xai.grok-4.7` from the US Regions and `global.xai.grok-4.7` from
any Region that serves it. `bedrock-mantle` does not serve it at all: the bare ID
returned 404 `The model 'xai.grok-4.7' does not exist` in `us-east-1` and `us-west-2` in
October 2026, where `xai.grok-4.6` answers on that same path in `us-west-2`. Grok 4.6's
`bedrock-mantle` section and its background mode therefore have no equivalent here. The
setup cell checks both profiles in your Region.

This notebook calls the **Responses** API on `bedrock-runtime`, with the OpenAI SDK at
`https://bedrock-runtime.{region}.amazonaws.com/openai/v1`. Chat Completions and
Converse are on the same endpoint (section 8 and section 9).

**Covered:** first call, streaming, multi-turn and server-side state, reasoning, tool
use, structured output, vision, Chat Completions, Converse, and IAM permissions.

## Setup

In [ ]:
import json
import re
import sys

import boto3
from aws_bedrock_token_generator import provide_token
import openai
from openai import OpenAI

sys.path.insert(0, "../_shared")
from bedrock import parse_json_lenient, safe_print

REGION = "us-east-1"
GROK = "us.xai.grok-4.7"  # the US geographic inference profile
GROK_GLOBAL = "global.xai.grok-4.7"

# bedrock-runtime's OpenAI-compatible path takes the same short-term Bedrock API key as
# bedrock-mantle. Reasoning calls can take minutes, so set a client timeout.
client = OpenAI(
    base_url=f"https://bedrock-runtime.{REGION}.amazonaws.com/openai/v1",
    api_key=provide_token(region=REGION),
    timeout=300,
)

bedrock = boto3.client("bedrock", region_name=REGION)
profiles = {
    p["inferenceProfileId"]
    for page in bedrock.get_paginator("list_inference_profiles").paginate()
    for p in page["inferenceProfileSummaries"]
}
for model in (GROK, GROK_GLOBAL):
    print(f"{model:21} {'available' if model in profiles else 'no such profile in ' + REGION}")


def answer_text(response) -> str:
    """The answer, without the \\confidence{N} marker Grok 4.7 sometimes ends it with."""
    return re.sub(r"\s*\\confidence\{\d+\}\s*$", "", response.output_text or "").strip()

## 1. First call

On `bedrock-runtime` name an inference profile as the model; the bare ID returns 400
`Invocation of model ID xai.grok-4.7 with on-demand throughput isn't supported`. Grok
4.7 reasons before it answers and spends output tokens doing it, so too small a
`max_output_tokens` returns HTTP 200 with `status` `incomplete` and no text; the cells
here give it 2,000.

Asked for one word, the model sometimes adds a sentence of explanation, so the cell
checks the answer for `Canberra` rather than comparing the whole string. In October 2026
runs it also sometimes ended an answer with a `\confidence{80}` marker, which
`answer_text` in the setup cell removes.

Every call in this notebook passes `store=False`. The Responses API stores each response
for 30 days by default, and AWS advises passing `store=False` on every request you do
not want retained. With `global.`, a stored response can be kept in any commercial
Region the profile routes to, so use `us.` when data residency matters.

In [ ]:
response = client.responses.create(
    model=GROK,
    input="What is the capital of Australia? Answer in one word.",
    max_output_tokens=2000,
    store=False,
)
answer = answer_text(response)
print(answer)
print("says Canberra:", "canberra" in answer.lower())
print(f"status={response.status}  output_tokens={response.usage.output_tokens}")
if response.status != "completed":
    print("incomplete:", getattr(response.incomplete_details, "reason", None))

## 2. Streaming

With `stream=True` the SDK yields typed events, and answer text arrives in
`response.output_text.delta` events. The reasoning is encrypted, so no reasoning deltas
arrive. The `\confidence{80}` marker from section 1 can arrive in the last deltas, so a
UI that shows deltas as they arrive shows it too. The cell prints the raw deltas, then
the assembled answer from the final event with the marker removed by `answer_text`.

In [ ]:
stream = client.responses.create(
    model=GROK,
    input="Count from 1 to 10, separated by commas.",
    max_output_tokens=2000,
    store=False,
    stream=True,
)
reasoning_deltas = 0
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type in ("response.reasoning.delta", "response.reasoning_text.delta"):
        reasoning_deltas += 1  # count them here; a UI could show them as they arrive
    elif event.type in ("response.completed", "response.incomplete"):
        print(f"\nstatus={event.response.status}  reasoning deltas={reasoning_deltas}")
        print("answer:", answer_text(event.response))  # the assembled text, marker removed

## 3. Multi-turn

The Responses API keeps no state between calls unless you ask it to. To hold a
conversation yourself, send it in `input`, including the model's output items as they
arrived. With `include=["reasoning.encrypted_content"]` the reasoning item carries an
encrypted blob that gives the next turn the model's earlier reasoning.

In [ ]:
messages = [{"role": "user", "content": "My favourite colour is teal. Reply with one word: noted."}]
first = client.responses.create(
    model=GROK, input=messages, include=["reasoning.encrypted_content"],
    max_output_tokens=2000, store=False,
)
messages += [item.model_dump(exclude_none=True) for item in first.output]  # reasoning and answer
messages.append({"role": "user", "content": "What is my favourite colour? One word."})
second = client.responses.create(
    model=GROK, input=messages, include=["reasoning.encrypted_content"],
    max_output_tokens=2000, store=False,
)

print("sent back:", [item.type for item in first.output])
print("turn 1   :", answer_text(first))
print("turn 2   :", answer_text(second))
print("recalled teal:", "teal" in answer_text(second).lower())

With `store=True` the service keeps the response, and `previous_response_id` continues
from it, so the next request carries only the new turn. On `bedrock-runtime`, chaining
from a response made with `store=False` returns 400 `Previous response with id '…' not
found.` A stored response is kept for 30 days, so the cell deletes it when it is done.

A response stored on `bedrock-runtime` cannot be read back or removed through the API:
`responses.retrieve` and `responses.delete` both returned 500 `internal_server_error` on
`/openai/v1` in October 2026 runs, for every model tried, while `bedrock-mantle` served
both. So prefer `store=False` and hold the conversation yourself, as the cell above does.
A response you do store ages out after 30 days.

In [ ]:
turn1 = client.responses.create(
    model=GROK,
    input="My favourite colour is teal. Reply with one word: noted.",
    max_output_tokens=2000,
    store=True,  # kept for 30 days unless deleted, so turn 2 can refer to it
)
turn2 = client.responses.create(
    model=GROK,
    input="What is my favourite colour? One word.",
    previous_response_id=turn1.id,
    max_output_tokens=2000,
    store=False,
)
answer = answer_text(turn2)
print("turn 2:", answer)
print("recalled teal:", "teal" in answer.lower())

# bedrock-runtime answered 500 to both retrieve and delete on /openai/v1 in October 2026
# runs, so a response stored here cannot be read back or removed through the API.
try:
    client.responses.delete(turn1.id)
    print("deleted the stored response")
except openai.InternalServerError as exc:
    print(f"delete refused: {exc.status_code} {exc.code}; it ages out after 30 days")

## 4. Reasoning

Grok 4.7 always reasons. The `effort` in `reasoning` can be `low`, `medium`, `high` or
`xhigh`, and a response echoes the effort it used: with no `effort` in the request it
came back `high` in October 2026 runs, so the default is the second most expensive
setting. `none`, `minimal` and `max` each returned 400 `Unsupported value`, so reasoning
cannot be switched off; the Things that differ table has the error text, which does not
agree with itself about which values are supported.

The reasoning is encrypted: the `reasoning` item holds `encrypted_content` and no
readable text. `usage.output_tokens_details.reasoning_tokens` counts the reasoning you
pay for.

In [ ]:
response = client.responses.create(
    model=GROK,
    input='A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Answer with the amount only.',
    reasoning={"effort": "medium"},
    include=["reasoning.encrypted_content"],
    max_output_tokens=4000,
    store=False,
)
reasoning = [item for item in response.output if item.type == "reasoning"]
answer = answer_text(response)
print("answer      :", answer)
print("says $0.05  :", "0.05" in answer)  # the answer can run long, so check it for the amount
print("items       :", [item.type for item in response.output])
blob = "".join(item.encrypted_content or "" for item in reasoning)
readable = sum(len(item.content or []) + len(item.summary or []) for item in reasoning)
print(f"reasoning   : {len(blob)} characters of encrypted_content, {readable} readable parts")
print("effort used :", response.reasoning.effort)  # echoed back by the service
print(f"reasoning_tokens={response.usage.output_tokens_details.reasoning_tokens}  status={response.status}")

## 5. Tool use

A Responses tool is flat: `name`, `description` and `parameters` sit at the top level,
where Chat Completions nests them under `function`. Each call arrives as a
`function_call` output item. Run it, send the call and a `function_call_output` item
with the same `call_id` back in `input`, and repeat until the model answers in text.
Check each call's arguments before running it: a call says the model wants a tool, and
says nothing about whether the arguments are usable.

In [ ]:
def lookup_inventory(sku: str) -> dict:
    """Stand-in for a real inventory service."""
    stock = {"A-100": 42, "B-200": 0}
    if sku.upper() not in stock:
        return {"sku": sku, "error": "unknown SKU"}
    return {"sku": sku, "quantity": stock[sku.upper()]}


TOOLS = [{
    "type": "function",
    "name": "lookup_inventory",
    "description": "Current stock level for a SKU.",
    "parameters": {
        "type": "object",
        "properties": {"sku": {"type": "string", "description": "SKU code, e.g. A-100"}},
        "required": ["sku"],
    },
}]
IMPLEMENTATIONS = {"lookup_inventory": lookup_inventory}


def run_tool(name: str, args: dict) -> dict:
    """Refuse a call that names an unknown tool or does not match the tool's parameters."""
    spec = next((t for t in TOOLS if t["name"] == name), None)
    if spec is None:
        return {"error": f"unknown tool {name}"}
    params = spec["parameters"]
    missing = [k for k in params["required"] if k not in args]
    unknown = [k for k in args if k not in params["properties"]]
    wrong_type = [k for k in args if k in params["properties"] and params["properties"][k].get("type") == "string"
                  and not isinstance(args[k], str)]
    if missing or unknown or wrong_type:
        return {"error": f"bad arguments: missing {missing}, unknown {unknown}, not a string {wrong_type}"}
    return IMPLEMENTATIONS[name](**args)

In [ ]:
items = [{"role": "user", "content": "How many units of SKU A-100 do we have?"}]
for _ in range(5):  # bounded, so a model that keeps calling tools cannot loop forever
    response = client.responses.create(
        model=GROK, input=items, tools=TOOLS, max_output_tokens=2000, store=False
    )
    calls = [item for item in response.output if item.type == "function_call"]
    if not calls:
        break
    for call in calls:
        args = parse_json_lenient(call.arguments)
        result = run_tool(call.name, args)
        print(f"tool call: {call.name}({args}) -> {result}")
        items.append({"type": "function_call", "call_id": call.call_id, "name": call.name,
                      "arguments": call.arguments})
        items.append({"type": "function_call_output", "call_id": call.call_id, "output": json.dumps(result)})
else:
    raise RuntimeError("no answer after 5 rounds of tool calls")

answer = answer_text(response)
print("answer:", answer)
print("says 42:", "42" in answer)

## 6. Structured output

`text.format` with a JSON Schema asks for output in that shape. Check `status` first,
because a cut-off answer is cut-off JSON, and parse leniently. A schema controls the
shape of the output and has no say over the values in it, so the cell checks each value
against the source text.

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {
        "invoice_id": {"type": "string"},
        "vendor": {"type": "string"},
        "total": {"type": "number"},
        "currency": {"type": "string"},
    },
    "required": ["invoice_id", "vendor", "total", "currency"],
    "additionalProperties": False,
}
SOURCE = "Invoice: INV-1042\nVendor: Acme Pty Ltd\nAmount due: AUD 1,280.50\nDue date: 30 October 2026"
EXPECTED = {"invoice_id": "INV-1042", "vendor": "Acme Pty Ltd", "total": 1280.5, "currency": "AUD"}

response = client.responses.create(
    model=GROK,
    input=("Extract invoice_id, vendor, total and currency from this invoice as JSON. Write the "
           f"total as a number without a thousands separator, so 2,345.60 becomes 2345.6.\n\n{SOURCE}"),
    text={"format": {"type": "json_schema", "name": "invoice", "strict": True, "schema": SCHEMA}},
    max_output_tokens=2000,
    store=False,
)
if response.status != "completed":
    raise RuntimeError(f"incomplete output ({getattr(response.incomplete_details, 'reason', response.status)})")

invoice = parse_json_lenient(response.output_text)
print(json.dumps(invoice, indent=2))
wrong = {k: invoice.get(k) for k, v in EXPECTED.items() if invoice.get(k) != v}
print("values match the source" if not wrong else f"values that do NOT match the source: {wrong}")

## 7. Vision

Send the image as a data URL in an `input_image` part, before the text part. The slide
comes from a public AWS talk and ships with this repository, so the answer is known: its
title is `Ingestion from database`, and the cell counts how many of its three callout
lines the model quotes.

In [ ]:
from bedrock import SLIDE_CALLOUTS, SLIDE_TITLE, keyword_recall, slide_data_url

response = client.responses.create(
    model=GROK,
    input=[{"role": "user", "content": [
        {"type": "input_image", "image_url": slide_data_url()},
        {"type": "input_text", "text": "Read this slide. Give its title, then quote the three green callout lines."},
    ]}],
    max_output_tokens=2000,
    store=False,
)
answer = answer_text(response)
hits, total = keyword_recall(answer, SLIDE_CALLOUTS)
print(answer)
print(f"\nstatus={response.status}")
print(f"title found: {SLIDE_TITLE.lower() in answer.lower()}   callouts found: {hits}/{total}")

## 8. Chat Completions on the same path

Grok 4.7 also serves Chat Completions on the same path, where the reasoning effort is
`reasoning_effort` rather than a nested object. The answer comes back without the
reasoning, and `usage.completion_tokens_details.reasoning_tokens` still counts the
reasoning you pay for.

In [ ]:
response = client.chat.completions.create(
    model=GROK,
    messages=[{"role": "user", "content": 'A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Answer with the amount only.'}],
    reasoning_effort="high",
    max_completion_tokens=4000,
)
message = response.choices[0].message
details = response.usage.completion_tokens_details
answer = (message.content or "").strip()
print("answer           :", answer)
print("says $0.05       :", "0.05" in answer)
print("message.reasoning:", f"{len(getattr(message, 'reasoning', None) or '')} characters")
print("reasoning_tokens :", getattr(details, "reasoning_tokens", None))
print("finish_reason    :", response.choices[0].finish_reason)

## 9. On `bedrock-runtime`: Converse

The same profile IDs work with Converse through boto3, which signs each request with
your AWS credentials. The reasoning arrives as a `reasoningContent` block holding
`redactedContent`, the Converse form of the encrypted trace, so read the content blocks
by type rather than by index.

In [ ]:
runtime = boto3.client("bedrock-runtime", region_name=REGION)
response = runtime.converse(
    modelId=GROK,
    messages=[{"role": "user", "content": [{"text": "What is the capital of Australia? Answer in one word."}]}],
    inferenceConfig={"maxTokens": 2000},
)
blocks = response["output"]["message"]["content"]
print("blocks    :", [next(iter(b)) for b in blocks])
print("reasoning :", [list(b["reasoningContent"]) for b in blocks if "reasoningContent" in b])
text = "".join(b["text"] for b in blocks if "text" in b)
text = re.sub(r"\s*\\confidence\{\d+\}\s*$", "", text).strip()  # the same marker as answer_text
print("answer    :", text)
print("says Canberra:", "canberra" in text.lower())
print("stopReason:", response["stopReason"])

## 10. IAM permissions

Calling through an inference profile needs `bedrock:InvokeModel` on the profile and on
the foundation model in each Region the profile routes to. For Grok 4.7 on
`bedrock-runtime` the model card adds your account's default project,
`arn:aws:bedrock:{region}:{account-id}:project/default`. The cell builds that policy
from the profile's own ARNs and prints it with the account ID replaced.

In [ ]:
account = boto3.client("sts").get_caller_identity()["Account"]
profile = bedrock.get_inference_profile(inferenceProfileIdentifier=GROK)

POLICY = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
        "Resource": [
            profile["inferenceProfileArn"],
            *(m["modelArn"] for m in profile["models"]),  # each Region the profile routes to
            f"arn:aws:bedrock:{REGION}:{account}:project/default",
        ],
    }],
}
safe_print(json.dumps(POLICY, indent=2))  # replaces the account ID before printing

## Things that differ

What trips people up on this family, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Reading the answer | `output[0]` is a reasoning item on every call, because the model always reasons | Read `output_text`, or pick output items by `type`. |
| Stored by default | `store` defaults to true, and the response is kept for 30 days | Pass `store=False` unless you will chain from the response. Deleting is refused here, as the next row says. |
| Reading back or deleting a stored response | 500 `internal_server_error` from `responses.retrieve` and `responses.delete` on `bedrock-runtime`'s `/openai/v1` (October 2026), for every model tried | Prefer `store=False` and keep the conversation yourself (section 3). A response you do store ages out after 30 days. |
| Answer cut short | HTTP 200 with `status` `incomplete` and `incomplete_details.reason` `max_output_tokens` | Raise `max_output_tokens`, and check `status` before using the text. |
| Structured output values | Valid JSON that holds a wrong value | A schema only controls the shape. Check the values you depend on. |
| Rate limits | 429 under load | Retry with backoff ([`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)). |
| Bare model ID on `bedrock-runtime` | 400 `Invocation of model ID xai.grok-4.7 with on-demand throughput isn't supported` | Name `us.xai.grok-4.7` or `global.xai.grok-4.7`. |
| `us.` outside the US | 400 `The provided model identifier is invalid.` (measured from `eu-central-1`) | Use `global.xai.grok-4.7`, which the setup cell checks for. |
| Not on `bedrock-mantle` | 404 `The model 'xai.grok-4.7' does not exist`, in `us-east-1` and `us-west-2` (October 2026) | Call `bedrock-runtime`. Grok 4.6 is the Grok on `bedrock-mantle` ([`02-grok-4-6`](02-grok-4-6.ipynb)). |
| Reasoning effort | `none`, `minimal` and `max` return 400 `Unsupported value`, and the errors disagree: for `none` and `max` the message says *"Supported values are: 'low', 'medium', 'high', and 'xhigh'"*, while for `minimal` it also lists `none` and `max` | Use `low`, `medium`, `high` or `xhigh`. |
| Default effort | A request with no `reasoning.effort` comes back with `reasoning.effort` `high` (October 2026) | Set `effort` explicitly when you care what you pay for reasoning. |
| Background mode | 400 `The background parameter is not supported.` | There is no `bedrock-mantle` route for this model, so background mode is not available for it. Poll your own job instead. |
| Web search | 400 ``Tool type 'web_search' is not supported for model `xai.grok-4.7`.`` | Search with your own tool and pass the results in. |
| Answer marker | An answer that ends in `\confidence{80}`, on the Responses API, in streaming deltas and through Converse | Remove it before display, as `answer_text` does. |
| Permissions | The profile alone is not enough on `bedrock-runtime`, per the model card | Allow the foundation models and your default project too (section 10). |

## Next

- Grok 4.6, on both endpoints: [`02-grok-4-6`](02-grok-4-6.ipynb)
- Grok 4.3, on `bedrock-mantle`: [`01-grok-4-3`](01-grok-4-3.ipynb)
- Converse and inference profiles in depth:
  [`00-foundations/04`](../00-foundations/04-bedrock-runtime-converse-and-profiles.ipynb)
- Other families on the Responses API:
  [`03-google-gemma/01`](../03-google-gemma/01-gemma4-end-to-end.ipynb) (Gemma 4),
  [`01-openai-gpt/01`](../01-openai-gpt/01-responses-api-core.ipynb) (OpenAI GPT)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)